In [10]:
# analysis
import numpy as np
import pandas as pd 

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell
import scanpy as sc
import liana as li
import maboss

Basic data loading tets

In [ ]:
# free data (3k PBMC cells)
adata = sc.datasets.pbmc3k()

  0%|          | 0.00/5.58M [00:00<?, ?B/s]

In [15]:
# Wczytanie danych
adata = sc.datasets.pbmc3k()

# Wstępna normalizacja
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ZAPISANIE DO .raw - to rozwiązuje Twój błąd!
adata.raw = adata

# Aby mieć po czym grupować ('louvain'), musimy zredukować wymiary i sklastrować dane
sc.pp.pca(adata)
sc.pp.neighbors(adata)
# IMPORTANT we need to use LEIDEN not 
sc.tl.leiden(adata)

# Teraz uruchamiamy LIANA+]

li.mt.rank_aggregate(adata, groupby='leiden', return_all_lrs=True)

ccc_results = adata.uns['liana_res']
print(ccc_results.head())

/tmp/ipykernel_225517/3359902750.py:14: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and 

      source target ligand_complex receptor_complex  lr_means  \
58914      3      8         S100A9            ITGB2  3.493192   
55873      3      6         S100A9             CD68  3.410479   
57393      3      7         S100A9             CD68  3.401547   
57394      3      7         S100A9            ITGB2  3.275019   
58909      3      8         S100A8            ITGB2  3.241278   

       cellphone_pvals  expr_prod  scaled_weight  lr_logfc  spec_weight  \
58914              0.0  10.304152       1.567280  3.084493     0.068996   
55873              0.0   9.498365       1.829962  3.201487     0.113388   
57393              0.0   9.411355       1.820177  3.197615     0.112349   
57394              0.0   8.178726       1.362322  2.621409     0.054764   
58909              0.0   9.238342       1.632559  3.245403     0.085683   

        lrscore  specificity_rank  magnitude_rank  
58914  0.968880      2.613269e-07    5.820697e-09  
55873  0.967629      4.868351e-08    2.328239e-08  
57

Minimalny pipeline w pythonie

In [7]:
import maboss
import pandas as pd

# 1. Wczytanie modelu wewnątrzkomórkowego (np. sieci regulacji genów dla naszej komórki)
# Plik .bnd definiuje logikę, a .cfg parametry symulacji
sim = maboss.load("moj_model.bnd", "moj_model.cfg")

# Załóżmy, że LIANA+ dała nam takie wagi (prawdopodobieństwa) obecności sygnału w 3 punktach w czasie:
# np. stężenie cytokiny TNF w środowisku wokół naszej komórki
time_series_environment = [0.1, 0.8, 0.3] 

# 2. Pętla czasowa symulująca interakcję ze środowiskiem
czas_kroku_dt = 10 

for t, external_signal in enumerate(time_series_environment):
    print(f"--- Krok czasowy {t} ---")
    
    # a) Aktualizacja informacji ze środowiska (wymuszamy prawdopodobieństwo aktywacji receptora)
    # [prawdopodobieństwo_0, prawdopodobieństwo_1]
    sim.network.set_istate('Receptor_TNF', [1 - external_signal, external_signal])
    
    # b) Uruchomienie symulacji na czas dt
    sim.update_parameters(max_time=czas_kroku_dt)
    result = sim.run()
    
    # c) Odczytanie stanu końcowego komórki
    last_states = result.get_last_states_probtraj()
    
    # d) Magia integracji: Używamy stanu końcowego jako stanu początkowego na kolejny krok!
    # Pobieramy prawdopodobieństwa stanów węzłów z końca symulacji
    # i nadpisujemy nimi model na następny obieg pętli (z wyjątkiem receptora, który sterowany jest z zewnątrz)
    for node in sim.network.keys():
        if node != 'Receptor_TNF':
            # Uproszczenie: w prawdziwym kodzie mapujesz wyciągnięte prawdopodobieństwo z tabeli last_states
            prob_active = get_node_prob_from_last_state(last_states, node) 
            sim.network.set_istate(node, [1 - prob_active, prob_active])

    print("Symulacja kroku zakończona. Komórka zaadaptowała się do nowego środowiska.")

FileNotFoundError: [Errno 2] No such file or directory: 'moj_model.bnd'